In [1]:
pip install pandas numpy scikit-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer

# ==========================================
# PHASE A: LOAD DATASET & DEFINE TARGET
# ==========================================
print("Loading 'student_dropout_dataset_v3.csv'...")
df = pd.read_csv('student_dropout_dataset_v3.csv')

# Encode 'Dropout' target variable if it's text
if df['Dropout'].dtype == 'object':
    le_target = LabelEncoder()
    df['Dropout'] = le_target.fit_transform(df['Dropout'])

# Define X (features) and y (target). Drop 'Student_ID' as it's not a feature.
X = df.drop(['Student_ID', 'Dropout'], axis=1)
y = df['Dropout']

# ==========================================
# PHASE B: DATA CLEANING & ENCODING
# ==========================================
print("\nStarting Data Cleaning and Encoding...")

numeric_cols = X.select_dtypes(include=['int64', 'float64']).columns
categorical_cols = X.select_dtypes(include=['object']).columns

numeric_imputer = SimpleImputer(strategy='median')
categorical_imputer = SimpleImputer(strategy='most_frequent')
categorical_encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_imputer, numeric_cols),
        ('cat', categorical_encoder, categorical_cols)
    ]
)

X_processed = preprocessor.fit_transform(X)
cat_names = preprocessor.named_transformers_['cat'].get_feature_names_out(categorical_cols)
feature_names = np.concatenate([numeric_cols, cat_names])
X_clean = pd.DataFrame(X_processed, columns=feature_names)

# ==========================================
# PHASE C: STRATIFIED SPLITTING & THE DIAGRAM
# ==========================================
print("\nPerforming Stratified Splitting...")

# Split 1: 40% Main Model (Set-1), 60% Hospitals remainder
X_set1, X_remainder, y_set1, y_remainder = train_test_split(
    X_clean, y, test_size=0.60, stratify=y, random_state=42
)

# Split 2: Divide remainder into Hospital 1 and Hospital 2
X_set2, X_set3, y_set2, y_set3 = train_test_split(
    X_remainder, y_remainder, test_size=0.50, stratify=y_remainder, random_state=42
)

# --- Hospital 1 (Set-2) Strict Diagram Counts ---
X_hosp1_train, X_hosp1_final_test, y_hosp1_train, y_hosp1_final_test = train_test_split(
    X_set2, y_set2, test_size=100, stratify=y_set2, random_state=42
)
X_hosp1_initial_test, _, y_hosp1_initial_test, _ = train_test_split(
    X_hosp1_final_test, y_hosp1_final_test, train_size=50, stratify=y_hosp1_final_test, random_state=123
)

# --- Hospital 2 (Set-3) Strict Diagram Counts ---
X_hosp2_train, X_hosp2_final_test, y_hosp2_train, y_hosp2_final_test = train_test_split(
    X_set3, y_set3, test_size=125, stratify=y_set3, random_state=42
)
X_hosp2_initial_test, _, y_hosp2_initial_test, _ = train_test_split(
    X_hosp2_final_test, y_hosp2_final_test, train_size=75, stratify=y_hosp2_final_test, random_state=123
)

# ==========================================
# PHASE D: DOCUMENTATION & EXPORT
# ==========================================
print("\n" + "="*40)
print("  REQUIRED DOCUMENTATION FOR SUBMISSION")
print("="*40)
print("\n--- Data Split Rows ---")
print(f"Total Original Dataset: {len(df)}")
print(f"Set-1 (Main Model Data): {len(X_set1)} rows")
print(f"Set-2 (Hospital 1 Total Data): {len(X_set2)} rows")
print(f"Set-3 (Hospital 2 Total Data): {len(X_set3)} rows")

print("\n--- Stratification Check (Dropout Distribution) ---")
print(f"Main Model Distribution:\n", y_set1.value_counts(normalize=True).to_dict())
print(f"Hospital 1 Distribution:\n", y_set2.value_counts(normalize=True).to_dict())
print(f"Hospital 2 Distribution:\n", y_set3.value_counts(normalize=True).to_dict())
print("="*40)

# Save everything for the Federated simulation
X_set1.assign(Dropout=y_set1).to_csv('main_cloud_data.csv', index=False)
X_hosp1_train.assign(Dropout=y_hosp1_train).to_csv('hosp1_train.csv', index=False)
X_hosp1_initial_test.assign(Dropout=y_hosp1_initial_test).to_csv('hosp1_test_50.csv', index=False)
X_hosp1_final_test.assign(Dropout=y_hosp1_final_test).to_csv('hosp1_test_100.csv', index=False)
X_hosp2_train.assign(Dropout=y_hosp2_train).to_csv('hosp2_train.csv', index=False)
X_hosp2_initial_test.assign(Dropout=y_hosp2_initial_test).to_csv('hosp2_test_75.csv', index=False)
X_hosp2_final_test.assign(Dropout=y_hosp2_final_test).to_csv('hosp2_test_125.csv', index=False)

print("\nSUCCESS! 7 CSV files generated.")